In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *
from pyspark.ml import Pipeline, Transformer

# Initialize Spark
spark = SparkSession.builder.appName("Comparison").getOrCreate()

# 1. Custom Transformer Class
class FeatureEngineeringTransformer(Transformer):
    """
    A custom transformer that applies SQL-equivalent feature engineering:
    1. Type casting
    2. Imputation
    3. Date diff calculation
    4. Window aggregation
    """
    def _transform(self, df):
        # Define the Window for rolling average (equivalent to ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)
        w_spec = Window.partitionBy("person_id") \
                       .orderBy("month_reference") \
                       .rowsBetween(-2, 0)
        
        return df \
            .withColumn("date_paid_cleaned", F.to_date(F.col("date_paid"))) \
            .withColumn("invoice_value_cleaned", F.coalesce(F.col("invoice_value"), F.lit(0.0))) \
            .withColumn("days_to_pay", F.datediff(F.col("date_paid_cleaned"), F.col("date_overdue"))) \
            .withColumn("rolling_avg_3m_invoice", F.mean("invoice_value_cleaned").over(w_spec))

# 2. Setup Dummy Data
data = [
    ('A', '2023-01-01', '2023-01-10', '2023-01-08', 100.0),
    ('A', '2023-02-01', '2023-02-10', '2023-02-12', 200.0),
    ('A', '2023-03-01', '2023-03-10', None,         300.0),
    ('A', '2023-04-01', '2023-04-10', '2023-04-09', 400.0),
    ('B', '2023-01-01', '2023-01-15', '2023-01-15', 50.0),
    ('B', '2023-02-01', '2023-02-15', '2023-02-20', None)
]

schema = StructType([
    StructField("person_id", StringType(), True),
    StructField("month_reference", StringType(), True), # Passed as string, implicitly castable to timestamp usually
    StructField("date_overdue", StringType(), True),
    StructField("date_paid", StringType(), True),
    StructField("invoice_value", DoubleType(), True)
])

df_spark = spark.createDataFrame(data, schema)
# Ensure base dates are actual date/timestamp types for the logic to hold
df_spark = df_spark \
    .withColumn("month_reference", F.to_timestamp("month_reference")) \
    .withColumn("date_overdue", F.to_date("date_overdue"))

# 3. Create and Run Pipeline
pipeline = Pipeline(stages=[FeatureEngineeringTransformer()])
model = pipeline.fit(df_spark) # Transformers are stateless, but fit() is required by Pipeline API
result_df = model.transform(df_spark)

# 4. Display
result_df.select("person_id", "month_reference", "invoice_value", "rolling_avg_3m_invoice", "days_to_pay").show()